In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import pandas as pd

from predql.converter import SConverter, TConverter

In [4]:
from relbench.datasets import Dataset
from relbench.datasets import get_dataset, get_dataset_names
from relbench.tasks import BaseTask
from relbench.tasks import get_task_names, get_task

get_dataset_names()

['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-mimic',
 'rel-trial',
 'rel-arxiv',
 'rel-salt',
 'rel-ratebeer',
 'dbinfer-avs',
 'dbinfer-mag',
 'dbinfer-diginetica',
 'dbinfer-retailrocket',
 'dbinfer-seznam',
 'dbinfer-amazon',
 'dbinfer-stackexchange',
 'dbinfer-outbrain-small']

In [5]:
def get_timestamps(dataset             : Dataset, 
                   timedelta           : pd.Timedelta, 
                   num_eval_timestamps : int,
                   split               : str) -> "pd.Series[pd.Timestamp]":
    db = dataset.get_db(upto_test_timestamp=(split != "test"))

    corrective_timedelta = pd.Timedelta("1ms")
    
    if split == "train":
        start = dataset.val_timestamp - timedelta + corrective_timedelta
        end = db.min_timestamp + corrective_timedelta
        freq = -timedelta
    elif split == "val":
        start = dataset.val_timestamp + corrective_timedelta
        end = min(
            dataset.val_timestamp
            + timedelta * (num_eval_timestamps - 1),
            dataset.test_timestamp - timedelta,
            ) + corrective_timedelta
        freq = timedelta
    elif split == "test":
        start = dataset.test_timestamp + corrective_timedelta
        end = min(
            dataset.test_timestamp
            + timedelta * (num_eval_timestamps - 1),
            db.max_timestamp - timedelta,
            ) + corrective_timedelta
        freq = timedelta
    else:
        pass

    timestamps = pd.date_range(start=start, end=end, freq=freq)
    return timestamps

In [6]:
def process_df_rb(df_rb     : pd.DataFrame, 
                  fk        : str, 
                  timestamp : str, 
                  label     : str) -> pd.DataFrame:
    renamed_df_rb = df_rb.rename(columns={fk        : 'fk',
                                          timestamp : 'timestamp',
                                          label     : 'label'})
    df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])
    
    corrective_timedelta = pd.Timedelta("1ms")
    df_rb['timestamp'] = df_rb['timestamp'] + corrective_timedelta

    return df_rb

In [7]:
def merge_dataframes(df_rb     : pd.DataFrame, 
                     df_predql : pd.DataFrame) -> None:
    # if pd.api.types.is_bool_dtype(df_predql['label']):
    #     df_predql['label'] = df_predql['label'].astype(int)
    
    merged = pd.merge(
    df_rb,       
    df_predql,   
    on=['fk', 'timestamp', 'label'], 
    how='outer',       
    suffixes=('_rb', '_predql'), 
    indicator=True    
    )

    print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
    print(f"Only in PredQL:\n {merged[merged['_merge'] == 'right_only']}")
    print(f"In both:\n {merged[merged['_merge'] == 'both']}")

In [8]:
def check_correctness(dataset            : Dataset, 
                      task               : BaseTask,
                      split              : str, 
                      predql_query       : str,
                      fk_col_name        : str,
                      timestamp_col_name : str,
                      label_col_name     : str) -> None:
    timestamps = get_timestamps(dataset, task.timedelta, task.num_eval_timestamps, split)
    
    print(f"TIMEDELTA: {task.timedelta}")
    print(f"NUM_EVAL_TIMESTAMPS: {task.num_eval_timestamps}")

    converter = TConverter(dataset.get_db(upto_test_timestamp=(split != "test")), timestamps)
    df_rb = task.get_table(split, mask_input_cols=False).df
    df_rb = process_df_rb(df_rb, fk_col_name, timestamp_col_name, label_col_name)
    df_predql = converter.convert(predql_query).df
    
    print(f"------------------- START {split.upper()} -------------------")
    merge_dataframes(df_rb, df_predql)
    print(f"------------------- END {split.upper()} ---------------------")

# F1 Database

In [9]:
dataset_f1 = get_dataset(name="rel-f1", download=True)
get_task_names("rel-f1")

['driver-position',
 'driver-dnf',
 'driver-top3',
 'driver-circuit-compete',
 'results-position',
 'qualifying-position',
 'constructor_results-points',
 'constructor_standings-position']

## Entity Classification Tasks

### driver-dnf

Task Description: For each driver predict the if they will DNF (did not finish) a race in the next 1 month. 

In [10]:
task_f1_dnf = get_task("rel-f1", "driver-dnf", download=False)

In [11]:
predql_query = """
    PREDICT MAX(results.statusId, 0, 30, DAYS) != 1
    FOR EACH drivers.driverId
    ASSUMING COUNT(results.*, -365, 0, DAYS) != 0
    WHERE MAX(results.statusId, 0, 30, DAYS) IS NOT NULL 
    ;
"""

In [12]:
# TRAIN

check_correctness(dataset=dataset_f1, 
                  task=task_f1_dnf, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.02 seconds.
TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START TRAIN -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                     timestamp   fk label _merge
0     2000-02-27 00:00:00.001    1     1   both
1     2000-03-28 00:00:00.001    1     1   both
2     2000-04-27 00:00:00.001    1     1   both
3     2000-05-27 00:00:00.001    1     1   both
4     2000-06-26 00:00:00.001    1     1   both
...                       ...  ...   ...    ...
11406 1950-08-18 00:00:00.001  802     0   both
11407 1950-05-20 00:00:00.001  803     1   both
11408 1953-05-04 00:00:00.001  804     1   both
11409 1954-05-29 00:00:00.001  805     1   both
11410 1956-01-19 00:00:00.001  806     1   both

[11411 rows x 4 columns]
------------------- E

In [13]:
# VAL

check_correctness(dataset=dataset_f1, 
                  task=task_f1_dnf, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START VAL -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                   timestamp  fk label _merge
0   2007-02-20 00:00:00.001   0     0   both
1   2007-03-22 00:00:00.001   0     0   both
2   2007-04-21 00:00:00.001   0     0   both
3   2007-05-21 00:00:00.001   0     0   both
4   2007-06-20 00:00:00.001   0     0   both
..                      ...  ..   ...    ...
561 2005-05-31 00:00:00.001  39     1   both
562 2005-06-30 00:00:00.001  39     1   both
563 2005-05-31 00:00:00.001  40     1   both
564 2005-08-29 00:00:00.001  41     1   both
565 2005-09-28 00:00:00.001  41     1   both

[566 rows x 4 columns]
------------------- END VAL ---------------------


In [14]:
# TEST

check_correctness(dataset=dataset_f1, 
                  task=task_f1_dnf, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.02 seconds.
TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START TEST -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                   timestamp   fk label _merge
0   2010-03-02 00:00:00.001    0     0   both
1   2010-04-01 00:00:00.001    0     0   both
2   2010-05-01 00:00:00.001    0     1   both
3   2010-05-31 00:00:00.001    0     0   both
4   2010-06-30 00:00:00.001    0     0   both
..                      ...  ...   ...    ...
697 2013-03-16 00:00:00.001  819     1   both
698 2013-03-16 00:00:00.001  820     1   both
699 2013-03-16 00:00:00.001  821     1   both
700 2013-03-16 00:00:00.001  822     1   both
701 2013-03-16 00:00:00.001  823     1   both

[702 rows x 4 columns]
------------------- END TEST -------------------

### driver-top3

Task Description: For each driver predict if they will qualify in the top-3 for a race in the next 1 month. 

In [15]:
task_f1_top3 = get_task("rel-f1", "driver-top3", download=False)

In [16]:
predql_query = """
    PREDICT MIN(qualifying.position, 0, 30, DAYS) <= 3
    FOR EACH drivers.driverId
    WHERE MIN(qualifying.position, 0, 30, DAYS) IS NOT NULL;
"""

In [17]:
# TRAIN

check_correctness(dataset=dataset_f1, 
                  task=task_f1_top3, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START TRAIN -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                    timestamp   fk label _merge
0    2000-02-27 00:00:00.001    1     0   both
1    2000-03-28 00:00:00.001    1     0   both
2    2000-09-24 00:00:00.001    1     0   both
3    2001-09-19 00:00:00.001    1     0   both
4    2002-02-16 00:00:00.001    1     0   both
...                      ...  ...   ...    ...
1348 1994-08-27 00:00:00.001  112     0   both
1349 1994-08-27 00:00:00.001  113     0   both
1350 1994-09-26 00:00:00.001  114     0   both
1351 1994-10-26 00:00:00.001  114     0   both
1352 1994-10-26 00:00:00.001  115     0   both

[1353 rows x 4 columns]
------------------- END TRAIN ---------------------


In [18]:
# VAL

check_correctness(dataset=dataset_f1, 
                  task=task_f1_top3, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START VAL -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                   timestamp  fk label _merge
0   2007-02-20 00:00:00.001   0     0   both
1   2007-03-22 00:00:00.001   0     1   both
2   2007-04-21 00:00:00.001   0     0   both
3   2007-05-21 00:00:00.001   0     1   both
4   2007-06-20 00:00:00.001   0     1   both
..                      ...  ..   ...    ...
583 2005-05-31 00:00:00.001  39     0   both
584 2005-06-30 00:00:00.001  39     0   both
585 2005-05-31 00:00:00.001  40     0   both
586 2005-08-29 00:00:00.001  41     0   both
587 2005-09-28 00:00:00.001  41     0   both

[588 rows x 4 columns]
------------------- END VAL ---------------------


In [19]:
# TEST

check_correctness(dataset=dataset_f1, 
                  task=task_f1_top3, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START TEST -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                   timestamp   fk label _merge
0   2010-03-02 00:00:00.001    0     0   both
1   2010-04-01 00:00:00.001    0     0   both
2   2010-05-01 00:00:00.001    0     1   both
3   2010-05-31 00:00:00.001    0     1   both
4   2010-06-30 00:00:00.001    0     0   both
..                      ...  ...   ...    ...
721 2013-03-16 00:00:00.001  819     0   both
722 2013-03-16 00:00:00.001  820     0   both
723 2013-03-16 00:00:00.001  821     0   both
724 2013-03-16 00:00:00.001  822     0   both
725 2013-03-16 00:00:00.001  823     0   both

[726 rows x 4 columns]
------------------- END TEST ---------------------


## Entity Regression Tasks

### driver-position

Task Description: Predict the average finishing position of each driver all races in the next 2 months. 

In [20]:
task_f1_pos = get_task("rel-f1", "driver-position", download=False)

In [21]:
predql_query = """
    PREDICT AVG(results.positionOrder, 0, 60, DAYS)
    FOR EACH drivers.driverId
    WHERE AVG(results.positionOrder, 0, 60, DAYS) IS NOT NULL;
"""

In [22]:
# TRAIN

check_correctness(dataset=dataset_f1, 
                  task=task_f1_pos, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START TRAIN -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                    timestamp   fk      label _merge
0    2000-01-28 00:00:00.001    1  14.000000   both
1    2000-03-28 00:00:00.001    1  17.666667   both
2    2000-05-27 00:00:00.001    1  13.750000   both
3    2000-07-26 00:00:00.001    1  15.000000   both
4    2000-09-24 00:00:00.001    1  19.500000   both
...                      ...  ...        ...    ...
7448 1950-06-19 00:00:00.001  801   6.000000   both
7449 1950-08-18 00:00:00.001  802   2.000000   both
7450 1953-04-04 00:00:00.001  804  17.000000   both
7451 1954-05-29 00:00:00.001  805  30.000000   both
7452 1956-01-19 00:00:00.001  806   6.000000   both

[7453 rows x 4 columns]
------------------- END TRAIN ---------------------


In [23]:
# VAL

check_correctness(dataset=dataset_f1, 
                  task=task_f1_pos, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START VAL -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                   timestamp   fk      label _merge
0   2007-02-20 00:00:00.001    0   2.333333   both
1   2007-04-21 00:00:00.001    0   1.500000   both
2   2007-06-20 00:00:00.001    0   4.000000   both
3   2007-08-19 00:00:00.001    0   6.200000   both
4   2007-10-18 00:00:00.001    0   7.000000   both
..                      ...  ...        ...    ...
494 2009-08-08 00:00:00.001  152  17.400000   both
495 2009-10-07 00:00:00.001  152  17.000000   both
496 2009-08-08 00:00:00.001  153  16.800000   both
497 2009-10-07 00:00:00.001  153  15.500000   both
498 2009-10-07 00:00:00.001  154   8.000000   both

[499 rows x 4 columns]
------------------- END VAL ---------------------


In [24]:
# TEST

check_correctness(dataset=dataset_f1, 
                  task=task_f1_pos, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------------------- START TEST -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                   timestamp   fk      label _merge
0   2010-03-02 00:00:00.001    0   4.250000   both
1   2010-05-01 00:00:00.001    0   4.600000   both
2   2010-06-30 00:00:00.001    0   8.666667   both
3   2010-08-29 00:00:00.001    0  10.000000   both
4   2010-10-28 00:00:00.001    0   3.000000   both
..                      ...  ...        ...    ...
755 2016-05-29 00:00:00.001  835  17.000000   both
756 2016-01-30 00:00:00.001  836  19.000000   both
757 2016-03-30 00:00:00.001  836  19.250000   both
758 2016-05-29 00:00:00.001  836  18.000000   both
759 2016-03-30 00:00:00.001  837  10.000000   both

[760 rows x 4 columns]
------------------- END TEST ---------------------


# Stack-Exchange Q&A Website Database

In [8]:
dataset_stack = get_dataset(name="rel-stack", download=False)
get_task_names("rel-stack")

['user-engagement',
 'post-votes',
 'user-badge',
 'user-post-comment',
 'post-post-related']

## Entity Classification Tasks

### user-engagement

Task Description: For each user predict if a user will make any votes, posts, or comments in the next 3 months. 

In [27]:
task_stack_engage = get_task("rel-stack", "user-engagement", download=False)

In [28]:
predql_query = """
     PREDICT COUNT(votes.Id, 0, 91, DAYS) != 0
          OR COUNT(posts.Id, 0, 91, DAYS) != 0
          OR COUNT(comments.Id, 0, 91, DAYS) != 0
     FOR EACH users.Id
     ASSUMING COUNT(votes.Id, -20, 0, YEARS) != 0
           OR COUNT(posts.Id, -20, 0, YEARS) != 0
           OR COUNT(comments.Id, -20, 0, YEARS) != 0;
"""

In [29]:
# TRAIN
# must be not filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_engage, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 6.50 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

------------------- START TRAIN -------------------
Only in RelBench:
                       timestamp      fk label     _merge
1249    2010-07-15 00:00:00.001      37     1  left_only
7790    2010-07-15 00:00:00.001     270     1  left_only
20439   2010-01-14 00:00:00.001     855     0  left_only
20440   2010-04-15 00:00:00.001     855     0  left_only
20441   2010-07-15 00:00:00.001     855     0  left_only
...                         ...     ...   ...        ...
1360845 2020-07-02 00:00:00.001  251501     0  left_only
1360846 2019-10-03 00:00:00.001  252540     0  left_only
1360847 2020-01-02 00:00:00.001  252540     0  left_only
1360848 2020-04-02 00:00:00.001  252540     0  left_only
1360849 2020-07-02 00:00:00.001  252540     0  left_only

[1613 rows x 4 columns]
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                       timestamp      fk label _merge
0       2010-10-14 00:00:00.001       0     0   both
1       2011-01-13 00

In [18]:
table_df = dataset_stack.get_db().table_dict['users'].df
table_df[table_df['Id'] == 37]

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 7.62 seconds.


,Id,AccountId,DisplayName,Location,ProfileImageUrl,WebsiteUrl,AboutMe,CreationDate
37,37,9084.0,hadley,"Houston, TX",NaN,http://hadley.nz,I'm an assistant professor of Statistics at Ri...,2010-07-19 19:13:09.870


In [81]:
# VAL
# must be not filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_engage, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------ASSUMING_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        predict.timestamp,
        predict.label
    FROM
        users parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.Id AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.Id,
            for_each.timestamp,
        FROM
            users parent
        JOIN
            (SELECT
            parent.Id AS fk,
            time.timestamp AS timestamp
        FROM
            users parent
        JOIN
            timestamp_df time
        ON
            time.timestamp >= parent.CreationDate
    ) for_each
        ON
            for_each.fk = parent.Id
    ------HELP_PART_END------
       

In [82]:
# TEST
# must be filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_engage, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------ASSUMING_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        predict.timestamp,
        predict.label
    FROM
        users parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.Id AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.Id,
            for_each.timestamp,
        FROM
            users parent
        JOIN
            (SELECT
            parent.Id AS fk,
            time.timestamp AS timestamp
        FROM
            users parent
        JOIN
            timestamp_df time
        ON
            time.timestamp >= parent.CreationDate
    ) for_each
        ON
            for_each.fk = parent.Id
    ------HELP_PART_END------
       

### user-badge

Task Description: For each user predict if a user will receive a new badge in the next 3 months. 

In [30]:
task_stack_badge = get_task("rel-stack", "user-badge", download=False)

In [31]:
predql_query = """
    PREDICT COUNT(badges.Id, 0, 91, DAYS) != 0
    FOR EACH users.Id;
"""

In [32]:
# TRAIN

check_correctness(dataset=dataset_stack, 
                  task=task_stack_badge, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------------------- START TRAIN -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                       timestamp      fk label _merge
0       2010-10-14 00:00:00.001       0     0   both
1       2011-01-13 00:00:00.001       0     0   both
2       2011-04-14 00:00:00.001       0     0   both
3       2011-07-14 00:00:00.001       0     0   both
4       2011-10-13 00:00:00.001       0     0   both
...                         ...     ...   ...    ...
3386271 2020-07-02 00:00:00.001  239940     0   both
3386272 2020-07-02 00:00:00.001  239941     0   both
3386273 2020-07-02 00:00:00.001  239942     0   both
3386274 2020-07-02 00:00:00.001  239943     0   both
3386275 2020-07-02 00:00:00.001  239944     1   both

[3386276 rows x 4 columns]
------------------- END TRAIN ---------------------


In [33]:
# VAL

check_correctness(dataset=dataset_stack, 
                  task=task_stack_badge, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------------------- START VAL -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                      timestamp      fk label _merge
0      2020-10-01 00:00:00.001       0     0   both
1      2020-10-01 00:00:00.001       1     0   both
2      2020-10-01 00:00:00.001       2     0   both
3      2020-10-01 00:00:00.001       3     0   both
4      2020-10-01 00:00:00.001       4     0   both
...                        ...     ...   ...    ...
247393 2020-10-01 00:00:00.001  247393     0   both
247394 2020-10-01 00:00:00.001  247394     1   both
247395 2020-10-01 00:00:00.001  247395     0   both
247396 2020-10-01 00:00:00.001  247396     0   both
247397 2020-10-01 00:00:00.001  247397     1   both

[247398 rows x 4 columns]
------------------- END VAL ---------------------


In [34]:
# TEST

check_correctness(dataset=dataset_stack, 
                  task=task_stack_badge, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 6.02 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------------------- START TEST -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                      timestamp      fk label _merge
0      2021-01-01 00:00:00.001       0     0   both
1      2021-01-01 00:00:00.001       1     0   both
2      2021-01-01 00:00:00.001       2     0   both
3      2021-01-01 00:00:00.001       3     0   both
4      2021-01-01 00:00:00.001       4     1   both
...                        ...     ...   ...    ...
255355 2021-01-01 00:00:00.001  255355     1   both
255356 2021-01-01 00:00:00.001  255356     0   both
255357 2021-01-01 00:00:00.001  255357     0   both
255358 2021-01-01 00:00:00.001  255358     1   both
255359 2021-01-01 00:00:00.001  255359     1   bot

## Entity Regression Tasks

### post-votes

Task Description: For each user post predict how many votes it will receive in the next 3 months 

In [35]:
task_stack_post_votes = get_task("rel-stack", "post-votes", download=False)

In [36]:
predql_query = """
    PREDICT COUNT_DISTINCT(votes.Id WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH posts.Id WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [37]:
# TRAIN

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_votes, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------------------- START TRAIN -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                       timestamp      fk  label _merge
0       2009-04-16 00:00:00.001       0      0   both
1       2009-07-16 00:00:00.001       0      0   both
2       2009-10-15 00:00:00.001       0      0   both
3       2010-01-14 00:00:00.001       0      0   both
4       2010-04-15 00:00:00.001       0      0   both
...                         ...     ...    ...    ...
2453916 2020-07-02 00:00:00.001  315139      0   both
2453917 2020-07-02 00:00:00.001  315140      0   both
2453918 2020-07-02 00:00:00.001  315142      0   both
2453919 2020-07-02 00:00:00.001  315144      0   both
2453920 2020-07-02 00:00:00.001  315145      0   both

[2453921 rows x 4 columns]
------------------- END TRAIN -------------

In [38]:
# VAL

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_votes, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------------------- START VAL -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                      timestamp      fk  label _merge
0      2020-10-01 00:00:00.001       0      4   both
1      2020-10-01 00:00:00.001      19      2   both
2      2020-10-01 00:00:00.001      23      2   both
3      2020-10-01 00:00:00.001      24      1   both
4      2020-10-01 00:00:00.001      25      1   both
...                        ...     ...    ...    ...
156211 2020-10-01 00:00:00.001  324972      0   both
156212 2020-10-01 00:00:00.001  324973      0   both
156213 2020-10-01 00:00:00.001  324976      0   both
156214 2020-10-01 00:00:00.001  324977      0   both
156215 2020-10-01 00:00:00.001  324980      0   both

[156216 rows x 4 columns]
------------------- END VAL ---------------------


In [39]:
# TEST

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_votes, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------------------- START TEST -------------------
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                      timestamp      fk  label _merge
0      2021-01-01 00:00:00.001       0      2   both
1      2021-01-01 00:00:00.001      19      0   both
2      2021-01-01 00:00:00.001      23      0   both
3      2021-01-01 00:00:00.001      24      0   both
4      2021-01-01 00:00:00.001      25      0   both
...                        ...     ...    ...    ...
160898 2021-01-01 00:00:00.001  333883      0   both
160899 2021-01-01 00:00:00.001  333885      0   both
160900 2021-01-01 00:00:00.001  333886      0   both
160901 2021-01-01 00:00:00.001  333887      0   both
160902 2021-01-01 00:00:00.001  333891      0   both

[160903 rows x 4 columns]
------------------- END TEST ---------------------


## Link Prediction Tasks

### user-post-comment

Task Description: Predict a list of existing posts that a user will comment in the next two months. 

In [9]:
task_stack_post_comm = get_task("rel-stack", "user-post-comment", download=False)

In [10]:
predql_query = """
    PREDICT LIST_DISTINCT(posts.Id WHERE posts.OwnerUserId != -1
                                     AND posts.OwnerUserId IS NOT NULL, 0, 91, DAYS)
    FOR EACH comments.UserId WHERE comments.UserId IS NOT NULL;
"""

In [ ]:
# TRAIN

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_comm, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 6.09 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


1. ERROR at line 4:22 - Column 'UserId' in FOR EACH clause is not a primary key column of table 'comments'
2. ERROR at line 2:26 - Table 'posts' in temporal aggregation is not connected to main table 'comments'


SystemExit: 1

/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [12]:
# VAL

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_comm, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
Making task table for val split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 5.58 seconds.
Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 5.92 seconds.
Done in 12.40 seconds.


1. ERROR at line 4:22 - Column 'UserId' in FOR EACH clause is not a primary key column of table 'comments'
2. ERROR at line 2:26 - Table 'posts' in temporal aggregation is not connected to main table 'comments'


SystemExit: 1

/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [13]:
# TEST

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_comm, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 6.06 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
Making task table for test split from scratch...
(You can also use `get_task(..., download=True)` for tasks prepared by the RelBench team.)
Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 6.75 seconds.
Done in 6.82 seconds.


1. ERROR at line 4:22 - Column 'UserId' in FOR EACH clause is not a primary key column of table 'comments'
2. ERROR at line 2:26 - Table 'posts' in temporal aggregation is not connected to main table 'comments'


SystemExit: 1

/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### post-post-related

Task Description: Predict a list of existing posts that users will link a given post to in the next two months.

In [123]:
# RelBench table

from relbench.tasks.stack import PostPostRelatedTask 

task_table_rb = PostPostRelatedTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
      timestamp  PostId postLinksIdList
0    2017-06-30   96362          [5578]
1    2017-06-30  156943        [111859]
2    2017-06-30  151830        [180081]
3    2017-06-30  139499  [10097, 37093]
4    2018-12-31  211907         [11654]
...         ...     ...             ...
4378 2016-03-31    2451         [31112]
4379 2018-06-30   46241         [26893]
4380 2014-06-30   27122         [53844]
4381 2017-12-31  209568         [13575]
4382 2018-12-31  114560          [2509]

[4383 rows x 3 columns],
  fkey_col_to_pkey_table={'PostId': 'posts', 'postLinksIdList': 'posts'},
  pkey_col=None,
  time_col=timestamp)


In [124]:
# PredQL table

# PredQL table
# NOTE: STATIC WHERE NEEDED

predql_query = """
    PREDICT LIST_DISTINCT(posts.Id, 1, 92, DAYS)
    FOR EACH users.Id;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    parent.Id AS fk,
    time.timestamp AS timestamp,
    main.comp_col
 AS label
FROM
    users parent
CROSS JOIN
    timestamp_df time
LEFT JOIN
    (------AGGREGATION_START------
    SELECT
        parent.Id AS fk,
        (
        SELECT
            ARRAY_AGG(freq_tbl.val ORDER BY freq_tbl.freq DESC)
        FROM (
            SELECT
                in_tbl.Id AS val,
                COUNT(*) AS freq
            FROM
                posts in_tbl
            WHERE
                in_tbl.CreationDate >= time.timestamp + INTERVAL '1 DAY'
            AND
                in_tbl.CreationDate <  time.timestamp + INTERVAL '92 DAY'
            AND
                in_tbl.OwnerUserId = parent.Id
            GROUP BY in_tbl.Id
            ) freq_tbl
        )
     AS comp_col,
        time.timestamp AS timestamp
    FROM
        users parent
    CROSS JOIN
        timestamp_df time
    LEFT JOIN
        posts aggr_tbl
    ON
        aggr_tbl.OwnerUserId = paren

In [125]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

In [126]:
df_rb = process_df_rb(df_rb, fk='PostId', timestamp='timestamp', label='postLinksIdList')

print(df_rb)
print(df_predql)

      timestamp      fk                          label
3130 2011-03-31      88                         [1574]
2283 2011-03-31     211                          [525]
1056 2011-03-31    2532                         [8069]
3250 2011-03-31    5236                         [6056]
2199 2011-03-31    5961                         [8166]
...         ...     ...                            ...
4168 2018-12-31  255382                        [59811]
461  2018-12-31  255385                 [78672, 53906]
2073 2018-12-31  255389                [122362, 93211]
3341 2018-12-31  255401                       [173128]
3307 2018-12-31  255418  [22174, 87268, 18138, 176509]

[4383 rows x 3 columns]
             fk  timestamp                           label
0             0 2011-03-31  [8976, 8974, 9221, 8973, 9222]
1             1 2011-03-31                            <NA>
2             2 2011-03-31                            <NA>
3             3 2011-03-31                            <NA>
4             4 2011

In [127]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
          timestamp      fk                       label_rb label_predql  \
8171520 2018-12-31  255382                        [59811]          NaN   
8171521 2018-12-31  255385                 [78672, 53906]          NaN   
8171522 2018-12-31  255389                [122362, 93211]          NaN   
8171523 2018-12-31  255401                       [173128]          NaN   
8171524 2018-12-31  255418  [22174, 87268, 18138, 176509]          NaN   

            _merge  
8171520  left_only  
8171521  left_only  
8171522  left_only  
8171523  left_only  
8171524  left_only  
Only in PredQL:
          timestamp      fk label_rb  \
0       2011-03-31       0      NaN   
1       2011-06-30       0      NaN   
2       2011-09-30       0      NaN   
3       2011-12-31       0      NaN   
4       2012-03-31       0      NaN   
...            ...     ...      ...   
8171515 2017-12-31  255359      NaN   
8171516 2018-03-31  255359      NaN   
8171517 2018-06-30  255359      NaN   
817